In [69]:
using TSSOS
using LinearAlgebra
using QuantumOptics
using DynamicPolynomials
using QuadGK
using JuMP
using Random
using NLopt

## Define the quantum system

In [70]:
# We look at a typical Hamiltonian

#Drift Hamiltonian

H0 = [
    0 0 0;
    0 0.515916 0;
    0 0 1
];

H0 ./= norm(H0, Inf)

# Control Hamiltonian
V = [
    0 0.707107 0;
    0.707107 0 1;
    0 1 0
]

V ./= norm(V, Inf);

# final time
const T = 0.5
m =8
dt = T/m

0.0625

In [71]:
H0*V-V*H0

3×3 Matrix{Float64}:
 0.0       -0.364808   0.0
 0.364808   0.0       -0.484084
 0.0        0.484084   0.0

## Useful functions with polynomials

In [72]:
function ∫(p::AbstractPolynomial, x::PolyVar, x_lower, x_upper)
    
    # get the index of the variable of integration
    ind_x = indexin([x], variables(p))[1]
        
    if isnothing(ind_x)
        # integration valuable is not found among vars
        return p * (x_upper - x_lower)
    end
    
    # get the indefinite integral
    int_p = sum(
        term * x * 1 // (exponents(term)[ind_x] + 1) for term in terms(p)
        init = 0 * x
    )
            
    # get the definite integral
    subs(int_p, x=>x_upper) - subs(int_p, x=>x_lower)
end

function ∫(M::AbstractMatrix, x::PolyVar, x_lower, x_upper)
   map(z -> ∫(z, x, x_lower, x_upper), M) 
end

function real_poly(p::Polynomial)
    #=
    Real part of the polynomial
    =#
    sum(
        real(c) * m for (c, m) in zip(coefficients(p), monomials(p))# if ~isapproxzero(abs(c))
    )
end

function square_frobenius_norm(M::AbstractArray)
    #=
    Square of the Frobenius norm of a matrix
    =#
    real_poly(sum(z' * z for z in M))
end

function square_frobenius_norm2(M::AbstractArray)
    #=
    Square of the Frobenius norm of a matrix
    =#
    sum(z' * z for z in M)
end

function trace_matrix(M::AbstractMatrix)
    d = size(M, 1)
    return sum(M[i, i] for i in 1:d)
end

function traceless_projection(M::AbstractMatrix)
    d = size(M, 1)
    trM = trace_matrix(M)
    return [M[i, j] - (i == j ? trM / d : zero(M[i, j])) for i in 1:d, j in 1:d]
end

traceless_projection (generic function with 1 method)

## Magnus expansion

In [73]:
@polyvar x[1:m]
@polyvar t[1:3]

function u(t, x)
    # the polynomial shape for control
    sum(x[n] * t^(n - 1) for n = 1:length(x))
end

function A(t, x,)
    #=
    The generator of motion entering the Magnus expansion
    =#
    (H0 + V * u(t, x)) / im
end

function commutator(a, b)
    a * b - b * a
end 

M = [-im * dt * (H0 + x[i] * V) for i in 1:m];

X = sum(M) ## first term

function commutator(A, B)
        return A * B - B * A
end

## second term
for i in 1:m
    for j in i:m
        X += 0.5 * commutator(M[i], M[j])
    end
end


## third term
for i in 1:m
    for j in i:m
        for k in j:m
        X += 1/6 * (commutator(M[i],commutator(M[j],M[k]))+commutator(M[k],commutator(M[j],M[i])))
        end
    end
end

## third term
for i in 1:m
    for j in i:m
        for k in j:m
            for l in k:m
                X += 1/12 *( commutator(commutator(commutator(M[i],M[j]),M[k]),M[l]) + commutator(M[i],commutator(commutator(M[j],M[k]),M[l])) + commutator(M[i],commutator(M[j],commutator(M[k],M[l]))) + commutator(M[j],commutator(M[k],commutator(M[l],M[i])))  )
            end
        
        end
    end
end

## Get random unitary target and take logarithm

In [74]:
function get_unitary(x::AbstractArray)
    #=
    Get the unitary given the coefficients for the polynomial control
    =#
    prod(exp(-im *dt* (H0 + xx * V)) for xx in x)
end

get_unitary (generic function with 1 method)

In [75]:
function logarithm_mat(U)
    Uc = ComplexF64.(U)
    F = eigen(Uc)
    Z = F.vectors
    n = 0
    theta_values = [log(complex(λ)) for λ in F.values]
    K = Diagonal(theta_values .+ 2π * im * n)
    Z * K * inv(Z)
end

logarithm_mat (generic function with 1 method)

## Functions to compare

In [76]:
function local_minimize(obj::AbstractPolynomial, init_x::AbstractArray)
    #=
    Perform local minimization of obj polynomial using init_x as initial guess
    =#
    vars = variables(obj)

    @assert length(vars) == length(init_x)
    
    function g(a...)
        # Converting polynomial expression to function to be minimize
        obj(vars => a)
    end
    
    model = Model(NLopt.Optimizer)

    set_optimizer_attribute(model, "algorithm", :LD_MMA)

    set_silent(model)
    @variable(model, y[1:length(vars)])

    # set initial guess
    for (var, init_val) in zip(y, init_x)
        set_start_value(var, init_val)
    end

    register(model, :g, length(y), g; autodiff = true)
    @NLobjective(model, Min, g(y...))
    JuMP.optimize!(model)

    map(value, y)
end

local_minimize (generic function with 1 method)

## TSSOS

In [77]:
using HDF5
@time begin

# Read targets from the pre-generated file
target_file = "targets.hdf5"
fid = h5open(target_file, "r")
U_targets = fid["U_targets"][:,:,:]
target_family_raw = fid["target_family"][]
target_family = split(target_family_raw, "\n")
planted_controls = fid["planted_controls"][:,:]
target_random_strength = fid["target_random_strength"][]
control_bound = fid["control_bound"][]
n_samples = fid["n_samples"][]
close(fid)

println("Loaded $n_samples targets from $target_file")

# Verify all targets are unitary
for i in 1:n_samples
    U = U_targets[i, :, :]
    @assert norm(U' * U - I) < 1e-10 "Target $i is not unitary"
end
println("All $n_samples targets verified unitary ✓")

# Baseline controls used only to report the objective before optimization.
initial_x = zeros(length(x), n_samples)
obj_initial_x = zeros(n_samples)
    
# the polynomial objective at min_x
glob_obj_min_x = zeros(n_samples)

# The global minimum via TSSOS library
tssos_glob_obj_min = zeros(n_samples)

# Frobenius norm difference between target and obtained unitaries
norm_U_target_minus_obtained = zeros(n_samples)

# The normalised overlap of the evolution and the target 
f_PSU = zeros(n_samples) 

Threads.@threads for i=1:n_samples
    
    # target unitary
    U_target = U_targets[i, :, :]
        
    # get the polynomial objective function
    Theta = logarithm_mat(U_target)

    # X is the anti-Hermitian truncated Magnus logarithm.
    # Convert to Hermitian generators: G = iΩ, then remove global phase.
    G_n = im * X
    G_star = im * Theta

    obj = square_frobenius_norm(
        traceless_projection(G_n - G_star)
    )
    
    # save the value of objective function for the baseline control
    obj_initial_x[i] = obj(initial_x[:, i])
  
    
    # Get the global minimum via TSSOS library with box constraints
    v = variables(obj)
    ineq_cons = [control_bound^2 - vi^2 for vi in v]
    pop = [obj; ineq_cons]
    d = div(maxdegree(obj) + 1, 2)
    opt,sol,data = tssos_first(pop, v, d; numeq=0, QUIET = true, solution = true)
    
    previous_sol = sol
    previous_opt = opt
    
    while ~isnothing(sol)
        previous_sol = sol
        previous_opt = opt
            
        opt,sol,data = tssos_higher!(data; QUIET = true, solution = true)
    end
    tssos_glob_obj_min[i] = previous_opt
    min_x = previous_sol
    
    # the polynomial objective at min_x
    glob_obj_min_x[i] = tssos_glob_obj_min[i]
        
    # get the Frobenius norm difference between target and obtained unitaries
    U_star = get_unitary(min_x)
    norm_U_target_minus_obtained[i] = norm(U_target - U_star)

    # Projective/process fidelity: |Tr(U_target' * U_star)|^2 / d^2
    d = size(U_star, 1)
    f_PSU[i] = real(abs(tr(U_target' * U_star))^2 / d^2)
end

end

Loaded 100 targets from targets.hdf5
All 100 targets verified unitary ✓
*********************************** TSSOS ***********************************
TSSOS is launching...
optimum = 0.001670853392483033
Global optimality certified with relative optimality gap 0.000007%!
No higher TS step of the TSSOS hierarchy!
*********************************** TSSOS ***********************************
TSSOS is launching...
optimum = 0.009447237259971904
Global optimality certified with relative optimality gap 0.000000%!
No higher TS step of the TSSOS hierarchy!
*********************************** TSSOS ***********************************
TSSOS is launching...
optimum = 0.0002593249283856622
Global optimality certified with relative optimality gap 0.000001%!
No higher TS step of the TSSOS hierarchy!
*********************************** TSSOS ***********************************
TSSOS is launching...
optimum = 0.004602281008260341
Global optimality certified with relative optimality gap 0.000011%!
No hi

In [78]:
using HDF5

filename = "results_$(m).hdf5"
h5open(filename, "w") do fid
    fid["U_targets"] = U_targets
    fid["target_family"] = join(target_family, "\n")
    fid["target_random_strength"] = target_random_strength
    fid["control_bound"] = control_bound
    fid["planted_controls"] = planted_controls
    fid["initial_x"] = initial_x
    fid["obj_initial_x"] = obj_initial_x
    fid["tssos_glob_obj_min"] = tssos_glob_obj_min
    fid["norm_U_target_minus_obtained"] = norm_U_target_minus_obtained
    fid["f_PSU"] = f_PSU
end;

In [79]:
println(f_PSU)

[0.9994422393100211, 0.9968780297984837, 0.9999150315231561, 0.9985039094710204, 0.9998529201156006, 0.9968782936586565, 0.999999427510939, 0.9985931976430363, 0.998336977466704, 0.9995327869005877, 0.9995835555788334, 0.9962152931175992, 0.9998685972772898, 0.997557099186706, 0.9994975531717678, 0.9990335340590816, 0.9948285998613451, 0.9995819422495011, 0.9967803622077298, 0.9954047034699877, 0.9998856877030291, 0.9998563336819262, 0.9998296916723977, 0.9995531116516053, 0.9979322337023222, 0.9960167590701405, 0.9988011020668733, 0.9990474401788174, 0.9999982685733053, 0.99952092087976, 0.9999848466351808, 0.9984243133659495, 0.9953411130661662, 0.9993564764806558, 0.9999626799600523, 0.9984395825828654, 0.99972984009132, 0.9970625221775626, 0.9992979856967472, 0.9998722278385311, 0.9993418801885618, 0.9999479813047538, 0.9964611271238222, 0.997852174305259, 0.9974461835603186, 0.9970538079189719, 0.9995546348516642, 0.9958776577083359, 0.9981204159616524, 0.9988557052804207, 0.99839